## Pruebas finales del código de `getting_data.ipynb`

### 1. Obtención de los catálogos

In [2]:
# ======================================================================================
# OBTENCIÓN DE DATOS (VERSIÓN FINAL Y CORREGIDA)
# ======================================================================================

import os
import json
import csv
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


# ======================================================================================
# DRIVER
# ======================================================================================

def crear_driver():
    """Crea y devuelve una instancia de Chrome en modo headless."""

    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    return webdriver.Chrome(options=opciones)


# Variable global del driver, se inicializa en control_flujo()
driver = None


# ======================================================================================
# BÚSQUEDA DE CATÁLOGO: EDITORIAL NORMAL (por páginas)
# ======================================================================================

def buscar_editorial(id_editorial, nombre_editorial, pag_inicio, pag_fin):
    """
    Recorre el catálogo de una editorial normal (hasta 200 páginas) iterando de pag_inicio a pag_fin.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **pag_inicio:** página del catálogo web donde empezar a hacer scraping
    * **pag_final:** última página del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionarios con los resultados del scraping
    """
    libros = []
    num_pagina = pag_inicio

    while num_pagina <= pag_fin:
        url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?page={num_pagina}"

        # Mostrar progreso solo hasta la página 10 para no llenar la consola
        if num_pagina < 10:
            print(f"Página {num_pagina}...")
        elif num_pagina == 10:
            print("Página 10 y más...")

        driver.get(url)
        time.sleep(1)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        elementos = soup.select("h2 a")

        # Si no hay libros en la página, se acabó el catálogo
        if not elementos:
            print("Sin más resultados.")
            break

        for h2 in soup.select("h2"):
            a = h2.find("a")
            if not a:
                continue

            titulo = a.get_text(strip=True)

            # El autor está en el h3 inmediatamente después del h2
            h3 = h2.find_next_sibling("h3")
            autor = h3.get_text(strip=True) if h3 else ""

            # El precio está en el primer <strong> después del h2
            etiqueta_precio = h2.find_next("strong")
            precio = etiqueta_precio.get_text(strip=True) if etiqueta_precio else ""

            url_libro = a["href"] if a.get("href") else ""

            libros.append({
                "editorial": nombre_editorial,
                "titulo": titulo,
                "autor": autor,
                "precio": precio,
                "url": url_libro,
            })

        num_pagina += 1
        time.sleep(0.5)

    return libros


# ======================================================================================
# BÚSQUEDA DE CATÁLOGO: EDITORIAL GRANDE (por años, para superar el límite de 200 págs)
# ======================================================================================

def buscar_editorial_grande(id_editorial, nombre_editorial, anio_inicio, anio_fin):
    """
    Recorre el catálogo de una editorial grande filtrando por año, lo que permite superar el límite de 200 páginas por búsqueda.
    Para cada año itera todas las páginas disponibles hasta que no haya resultados.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **anio_inicio:** página (del año) del catálogo web donde empezar a hacer scraping
    * **anio_final:** última página (del año) del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionario con los resultados del scraping
    """
    libros = []

    for anio in range(anio_inicio, anio_fin + 1):
        print(f"Año {anio}...")
        num_pagina = 1

        while True:
            url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?anios={anio}&page={num_pagina}"
            driver.get(url)
            time.sleep(1)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            elementos = soup.select("h2 a")
            
            # Si no hay libros, este año ya no tiene más páginas
            if not elementos:
                print(f"Sin más resultados en {anio}, página {num_pagina}.")
                break

            for h2 in soup.select("h2"):
                a = h2.find("a")
                if not a:
                    continue

                titulo = a.get_text(strip=True)

                h3 = h2.find_next_sibling("h3")
                autor = h3.get_text(strip=True) if h3 else ""

                etiqueta_precio = h2.find_next("strong")
                precio = etiqueta_precio.get_text(strip=True) if etiqueta_precio else ""

                url_libro = a["href"] if a.get("href") else ""

                libros.append({
                    "editorial": nombre_editorial,
                    "titulo": titulo,
                    "autor": autor,
                    "precio": precio,
                    "url": url_libro,
                })

            num_pagina += 1
            time.sleep(0.5)

    return libros


# ======================================================================================
# EXTRACCIÓN DE FICHA TÉCNICA Y SINOPSIS
# ======================================================================================

def extraer_datos(url):
    """
    Accede a la ficha de un libro y extrae todos los datos técnicos (ISBN, páginas, formato, etc.) y la sinopsis completa.
    Devuelve un diccionario con todos los campos encontrados.

    Parámetros:
    * **url:** link de la página de la ficha técnica

    Outputs: 
    * Diccionario con los resultados del scraping
    """
    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # La ficha técnica está dentro de elementos <dl class="datos-tecnicos">
    secciones = soup.find_all("dl", class_="datos-tecnicos")

    nombres = []  # nombres de los campos (dt)
    datos = []    # valores de los campos (dd)

    for seccion in secciones:
        etiquetas_nombre = seccion.find_all("dt")
        etiquetas_dato = seccion.find_all("dd")

        # Extraer nombres de los campos técnicos
        for nombre in etiquetas_nombre:
            nombres.append(nombre.get_text(strip=True).replace(":", "").strip())

        # Extraer valores de los campos técnicos
        # Hay tres posibles estructuras dentro de cada <dd>:
        for dato in etiquetas_dato:

            # Caso 1: el valor está en uno o varios <a> (ej: categorías, editorial)
            enlaces = dato.find_all("a")
            if enlaces:
                lista_valores = [enlace.get_text(strip=True) for enlace in enlaces]
                # Si hay un solo valor lo guardamos como string, si hay varios como lista
                datos.append(lista_valores[0] if len(lista_valores) == 1 else lista_valores)
                continue

            # Caso 2: el valor está en un <span> (ej: idioma)
            span = dato.find("span")
            if span:
                datos.append(span.get_text(strip=True))
                continue

            # Caso 3: el valor está directamente en el <dd> (ej: dimensiones, páginas)
            # Usamos split/join para limpiar espacios y saltos de línea extra
            texto = dato.get_text()
            datos.append(" ".join(texto.split()))

    # Sinopsis completa: está en un div separado fuera de la ficha técnica
    sinopsis = soup.find("div", id="collapseSynopsis")
    nombres.append("Sinopsis")
    if sinopsis:
        # Extraemos párrafo a párrafo y los unimos con " | "
        parrafos = [p.get_text(strip=True) for p in sinopsis.find_all("p")]
        datos.append(" | ".join(parrafos))
    else:
        datos.append("Sin sinopsis")

    # Construir diccionario emparejando cada nombre con su dato
    ficha_tecnica = {}
    for nombre, dato in zip(nombres, datos):
        ficha_tecnica[nombre] = dato

    return ficha_tecnica


# ======================================================================================
# SCRAPING COMPLETO DE UNA EDITORIAL (búsqueda + fichas + guardado CSV)
# ======================================================================================

def scrapear_editorial(id_editorial, nombre_editorial, inicio, fin, es_grande):
    """
    Orquesta el proceso completo para una editorial:
    1. Busca todos los libros del catálogo en el intervalo indicado.
    2. Entra en cada ficha técnica y extrae los datos.
    3. Guarda el resultado en un CSV en la carpeta 'data/'.

    Parámetros:
    * **id_editorial:** identificador de la URL (ej: "debolsillo_179709")
    * **nombre_editorial:** nombre legible (ej: "DEBOLSILLO")
    * **inicio:** página o año de inicio
    * **fin:** página o año de fin
    * **es_grande:** True si es editorial grande (búsqueda por años)

    Output:
    * Archivo .csv con las fichas técnicas de los libros
    """
    print(f"\n{'='*60}")
    print(f"Editorial: {nombre_editorial}")
    print(f"{'='*60}")

    # Fase 1: recoger listado de libros
    print("Leyendo catálogo...")
    if es_grande:
        libros = buscar_editorial_grande(id_editorial, nombre_editorial, inicio, fin)
    else:
        libros = buscar_editorial(id_editorial, nombre_editorial, inicio, fin)

    print(f"✓ {len(libros)} libros encontrados.")

    if not libros:
        print("No hay libros que procesar.")
        return

    # Fase 2: extraer fichas técnicas
    print("Extrayendo fichas técnicas...")
    lista_libros = []

    for i, libro in enumerate(libros):
        print(f"  [{i+1}/{len(libros)}] {libro['titulo'][:50]}...")

        ficha = extraer_datos(libro["url"] + "#fichaTecnica")

        # Añadir campo traducción si no existe
        if "Traducción" not in ficha:
            ficha["Traducción"] = "Sin traducción"

        # Añadir datos del catálogo a la ficha
        ficha["Título"] = libro["titulo"]
        ficha["Precio"] = libro["precio"]
        ficha["URL"] = libro["url"]

        lista_libros.append(ficha)

    # Fase 3: guardar JSON
    os.makedirs("data", exist_ok=True)
    ruta_json = f"data/bronze/catalogos/catalogo_{nombre_editorial.lower()}.json"
    campos = lista_libros[0].keys()

    # Revisión del catálogo
    if os.path.exists(ruta_json):
        with open(ruta_json, "r", encoding="utf-8") as f:
            libros_existentes = json.load(f)
    else:
        libros_existentes = []

    # Añadir nuevos libros al final
    libros_existentes.extend(lista_libros)

    # Guardar el resultado completo
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(libros_existentes, f, ensure_ascii=False, indent=2)

    print(f"✓ {len(lista_libros)} fichas guardadas en {ruta_json}.")


# ======================================================================================
# CONTROL DE FLUJO INTERACTIVO
# ======================================================================================

def control_flujo(ruta_editoriales="data/json/editoriales.json", ruta_estado="data/json/estado.json"):
    """
    Función principal que gestiona el flujo interactivo del scraping.

    Lee el fichero de editoriales (editoriales.json) con la información de cada una,
    consulta el estado guardado (estado.json) para saber cuáles ya están procesadas,
    y pregunta al usuario qué hacer con cada una.

    Estructura esperada de editoriales.json:
    {
        "debolsillo": {
            "id": "debolsillo_179709",
            "grande": false,
            "intervalo_max": [1, 200]   <- páginas si normal, años si grande
        },
        "espasa": {
            "id": "espasa_76490",
            "grande": true,
            "intervalo_max": [1990, 2024]
        }
    }

    Estructura de estado.json (se genera automáticamente):
    {
        "debolsillo": {
            "ultimo": 200    <- última página/año procesada
        }
    }
    """
    global driver

    # Cargar información de editoriales
    if not os.path.exists(ruta_editoriales):
        print(f"Error: no se encuentra '{ruta_editoriales}'.")
        return

    with open(ruta_editoriales, "r", encoding="utf-8") as f:
        info_editoriales = json.load(f)

    # Cargar estado previo si existe, o empezar desde cero
    if os.path.exists(ruta_estado):
        with open(ruta_estado, "r", encoding="utf-8") as f:
            estado = json.load(f)
    else:
        estado = {}

    # Iniciar el driver una sola vez para toda la sesión
    print("Iniciando navegador...")
    driver = crear_driver()
    print("✓ Navegador listo.\n")

    try:
        for nombre, info in info_editoriales.items():
            maximo = info["intervalo"][1]
            es_grande = info["grande"]
            id_editorial = info["id"]

            # Comprobar si ya está completamente procesada
            if nombre in estado and estado[nombre]["ultimo"] >= maximo:
                print(f"{nombre}: ya procesada completamente, saltando...")
                continue

            # Determinar punto de inicio (desde el principio o desde donde se dejó)
            if nombre in estado:
                ultimo_guardado = estado[nombre]["ultimo"]
                inicio_sugerido = ultimo_guardado+1 # retomamos desde el último guardado
                msg = f"{nombre}: proceso iniciado. Último guardado en {ultimo_guardado}. ¿Continuar? [Y/N]: "
            else:
                inicio_sugerido = info["intervalo"][0]
                msg = f"{nombre}: aún no procesada. ¿Comenzar? [Y/N]: "

            respuesta = input(msg).strip().upper()

            if respuesta == "N":
                print(f"Saltando {nombre}.\n")
                continue

            elif respuesta == "Y":
                # Pedir intervalo al usuario
                tipo = "año" if es_grande else "página"
                print(f"Inicio sugerido: {inicio_sugerido} | Máximo disponible: {maximo}")

                entrada_inicio = input(f"Introduce {tipo} de inicio [{inicio_sugerido}]: ").strip()
                entrada_fin = input(f"Introduce {tipo} de fin (máx. {maximo}): ").strip()

                # Usar valores sugeridos si el usuario no introduce nada
                inicio = int(entrada_inicio) if entrada_inicio else inicio_sugerido
                fin = min(int(entrada_fin), maximo)  # nunca superar el máximo

                # Ejecutar el scraping
                scrapear_editorial(id_editorial, nombre, inicio, fin, es_grande)

                # Actualizar y guardar estado
                if nombre not in estado:
                    estado[nombre] = {}
                estado[nombre]["ultimo"] = fin

                with open(ruta_estado, "w", encoding="utf-8") as f:
                    json.dump(estado, f, ensure_ascii=False, indent=2)

                print(f"Estado guardado: {nombre} → hasta {tipo} {fin}.\n")

            else:
                print("Respuesta no válida, saltando.\n")

    finally:
        # Cerrar el navegador siempre, aunque haya errores
        driver.quit()
        print("\nNavegador cerrado.")


# ======================================================================================
# PUNTO DE ENTRADA
# ======================================================================================

# if __name__ == "__main__":
#     control_flujo()

In [4]:
# ======================================================================================
# NUEVO SCRAPING
# ======================================================================================

def buscar_editorial(id_editorial, nombre_editorial, pag_inicio, pag_fin):
    """
    Recorre el catálogo de una editorial normal (hasta 200 páginas) iterando de pag_inicio a pag_fin.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **pag_inicio:** página del catálogo web donde empezar a hacer scraping
    * **pag_final:** última página del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionarios con los resultados del scraping
    """
    libros = []
    num_pagina = pag_inicio

    while num_pagina <= pag_fin:
        url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?page={num_pagina}"

        # Mostrar progreso solo hasta la página 10 para no llenar la consola
        if num_pagina < 10:
            print(f"Página {num_pagina}...")
        elif num_pagina == 10:
            print("Página 10 y más...")

        driver.get(url)
        time.sleep(1)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        elementos = soup.find_all("article", class_="book-col")

        for el in elementos:
            titulo = el.select_one("p.title").get_text(strip=True)
            autor = el.select_one("p.author").get_text(strip=True)
            precio = el.select_one("div.prices").get_text(strip=True)
            url_libro = el.select_one("p.title a")['href']
        

            libros.append({
                "editorial": nombre_editorial,
                "titulo": titulo,
                "autor": autor,
                "precio": precio,
                "url": url_libro,
            })

        num_pagina += 1
        time.sleep(0.5)

    return libros


# ======================================================================================
# EDITORIAL GRANDE NUEVO
# ======================================================================================

def buscar_editorial_grande(id_editorial, nombre_editorial, anio_inicio, anio_fin):
    """
    Recorre el catálogo de una editorial grande filtrando por año, lo que permite superar el límite de 200 páginas por búsqueda.
    Para cada año itera todas las páginas disponibles hasta que no haya resultados.
    Devuelve los diccionarios con título, autor, precio y URL de cada libro.

    Parámetros:
    * **id_editorial:** el id asignado a la editorial
    * **nombre_editorial:** el nombre de la editorial
    * **anio_inicio:** página (del año) del catálogo web donde empezar a hacer scraping
    * **anio_final:** última página (del año) del catálogo donde hacer scraping

    Outputs: 
    * Lista de diccionario con los resultados del scraping
    """
    libros = []

    for anio in range(anio_inicio, anio_fin + 1):
        print(f"Año {anio}...")
        num_pagina = 1

        while True:
            url = f"https://www.todostuslibros.com/editoriales/{id_editorial}/catalogo?anios={anio}&page={num_pagina}"
            driver.get(url)
            time.sleep(1)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            elementos = soup.find_all("article", class_="book-col")
    
            for el in elementos:
                titulo = el.select_one("p.title").get_text(strip=True)
                autor = el.select_one("p.author").get_text(strip=True)
                precio = el.select_one("div.prices").get_text(strip=True)
                url_libro = el.select_one("p.title a")['href']
        
    
                libros.append({
                    "editorial": nombre_editorial,
                    "titulo": titulo,
                    "autor": autor,
                    "precio": precio,
                    "url": url_libro,
                })

            num_pagina += 1
            time.sleep(0.5)

    return libros


control_flujo()

Iniciando navegador...
✓ Navegador listo.

Planeta: ya procesada completamente, saltando...
Espasa: ya procesada completamente, saltando...
Seix Barral: ya procesada completamente, saltando...
Inicio sugerido: 76 | Máximo disponible: 107

Editorial: Destino
Leyendo catálogo...
✓ 250 libros encontrados.
Extrayendo fichas técnicas...
  [1/250] CIEN RECETAS DE COMIDA CHINA.....DL...
  [2/250] Notícia de Catalunya...
  [3/250] La huída del tiempo...
  [4/250] El hermano Francisco...
  [5/250] HUGHES Y EL ONCE NEGRO ..........DL...
  [6/250] SOCIALISMO-LOS ORIGENES A 1875(2)DL...
  [7/250] Testament a Praga...
  [8/250] Homenots...
  [9/250] JO LES VOLIA...
  [10/250] FINS AL CEL...
  [11/250] MUNIA I LA SENYORA SETSIVELLES...
  [12/250] CARTES D'ITALIA...
  [13/250] REGOCIJO EN EL HOMBRE...
  [14/250] Un extraño en mi tumba...
  [15/250] La ciudad contra Kowalsyk...
  [16/250] El entierro de la sardina...
  [17/250] Por esos mundos...
  [18/250] SOMBRAS Y BULTOS.................DL...
  [19

Fin de capa **bronze**

### 2. Creación del DataFrame base para guardar en parquet

In [44]:
# ======================================================================================
# CREACIÓN DEL DATAFRAME BASE
# ======================================================================================

import json
import pandas as pd
import numpy as np
from pathlib import Path


def crear_df(ruta_catalogos="data/bronze/catalogos"):
    path = Path(ruta_catalogos)
    df = pd.DataFrame({})
    jsons = []

    print("="*50,"\nCreando DataFrame con todos los libros\n","="*50)
    for archivo in path.iterdir():
        if archivo.is_file():
            print(f"Añadiendo {archivo.name}")
            editorial = pd.read_json(archivo.absolute())
            jsons.append(editorial)

    df = pd.concat(jsons, axis=0)

    # Borrar filas repetidas
    print("Catálogos convertidos a DataFrame. Eliminando filas duplicadas...")
    df.drop_duplicates(subset=['EAN'], keep='first', inplace=True)

    # Limpiado de nombre de columnas
    df.columns = df.columns.str.strip().str.lower().str.translate(str.maketrans({"á": "a", "é": "e", "í": "i", "ó":"o", "ú": "u", "º": "", " ": "_"}))

    return df
    

    print("DataFrame creado con éxito.")

# def limpiar_df está abajo

def crear_parquet(df: pd.DataFrame, ruta_guardado="data/parquet/lista_libros.parquet"):

    # Creando .parquet
    df.to_parquet(ruta_guardado, index=False)
    print("Archivo .parquet creado con éxito.")

In [ ]:
import re
import numpy as np
import pandas as pd

# =============================================================================
# COLUMNAS
# =============================================================================

TRADUCTOR_EDITOR = [
    "traduccion",
    "edicion_literaria",
    "edicion",
    "direccion_de_edicion",
    "edicion_y_traduccion",
]

OTROS_CONTRIBUIDORES = [
    "epilogo",
    "prologo",
    "trabajo_preliminar",
    "contribucion",
    "introduccion",
    "comentarios_a_la_traduccion",
    "introduccion_a_notas",
    "prefacio",
    "notas",
    "compilacion",
]

ILUSTRACIONES = [
    "ilustracion",
    "ilustracion_fotografica",
    "fotografia",
]

ESCOLARES = [
    "material_enseñanza",
    "tipo_material_enseñanza",
    "tipo_enseñanza",
    "asignatura",
    "ciclo",
]

CATEGORIAS = [
    "categorias",
    "pais_de_publicacion",
    "asignatura",
    "tipo_material_enseñanza",
    "ciclo",
]

COLUMNAS_LISTA = list(set(
    TRADUCTOR_EDITOR
    + OTROS_CONTRIBUIDORES
    + ILUSTRACIONES
    + ESCOLARES
    + CATEGORIAS
    + ["autoria"]
))

COLUMNAS_FINALES = [
    "isbn",
    "ean",
    "titulo",
    "editorial",
    "id_editorial",
    "coleccion",
    "autoria",
    "traductor_y_editor",
    "otros_contribuidores",
    "subcategorias",
    "idioma_original",
    "idioma_de_publicacion",
    "fecha_publicacion",
    "n_paginas",
    "precio",
    "alto_mm",
    "ancho_mm",
    "grueso",
    "peso",
    "encuadernacion",
    "tipo_edicion",
    "presentacion",
    "es_ilustrado",
    "score_colaboradores",
    "colaboradores_destacados",
    "sinopsis",
    "url",
    "img",
]

# =============================================================================
# FUNCIONES AUXILIARES
# =============================================================================

def normalizar_lista(valor):
    """
    Convierte cualquier valor en una lista.

    NaN -> []
    str -> [str]
    list -> list limpia
    ndarray -> list
    """

    if valor is None:
        return []

    if isinstance(valor, float) and np.isnan(valor):
        return []

    if isinstance(valor, str):
        valor = valor.strip().capitalize()
        if valor == "":
            return []

        return [valor]

    if isinstance(valor, np.ndarray):
        valor = valor.tolist()

    if isinstance(valor, (list, tuple)):
        salida = []
        for x in valor:
            if pd.isna(x):
                continue

            x = str(x).strip().capitalize()

            if x:
                salida.append(x)

        return list(dict.fromkeys(salida))

    return [str(valor)]


def normalizar_columnas_lista(df):

    df = df.copy()

    for col in COLUMNAS_LISTA:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_lista)

    return df


# =============================================================================
# LIMPIEZA BÁSICA
# =============================================================================

def limpieza_basica(df,dict_editoriales=None,dict_encuadernacion=None,):

    df = df.copy()

    # quitar duplicados por EAN
    df.drop_duplicates(subset="ean", inplace=True)

    # eliminar libros sin autor y sin categoría
    df = df.dropna(
        subset=["autoria", "categorias"],
        how="all",
    )

    # fecha
    df["fecha_publicacion"] = pd.to_datetime(
        df["fecha_publicacion"],
        format="%d-%m-%Y",
        errors="coerce",
    )

    # sinopsis
    df["sinopsis"] = df["sinopsis"].fillna("Sin sinopsis")

    # ids editoriales
    if dict_editoriales is not None:
        df["id_editorial"] = df["editorial"].map(dict_editoriales)

    # encuadernación
    if dict_encuadernacion is not None:
        df["encuadernacion"] = df["encuadernacion"].map(dict_encuadernacion)

    return df


# =============================================================================
# MERGE DE COLUMNAS
# =============================================================================

def merge_columnas(df, nombre, columnas):

    df = df.copy()

    for col in columnas:
        if col not in df.columns:
            df[col] = [[] for _ in range(len(df))]

    df[nombre] = df[columnas].sum(axis=1).apply(lambda x: list(dict.fromkeys(x)))

    return df


# =============================================================================
# COLABORADORES DESTACADOS
# =============================================================================

def contador_nombres(df, min_apariciones=5):

    df = df.copy()

    todos = pd.concat([
        df["traductor_y_editor"].explode(),
        df["otros_contribuidores"].explode()
    ]).dropna()

    frecuencia = todos.value_counts()

    destacados = frecuencia[frecuencia >= min_apariciones].to_dict()

    def calcular_score(personas):
        return sum(
            destacados.get(p, 0)
            for p in personas
        )

    def colaboradores_destacados(personas):
        return [
            p
            for p in personas
            if p in destacados
        ]

    colaboradores = df["traductor_y_editor"]+ df["otros_contribuidores"]

    df["score_colaboradores"] = colaboradores.apply(calcular_score)

    df["colaboradores_destacados"] = colaboradores.apply(colaboradores_destacados)

    return df


# =============================================================================
# EXTRACCIÓN NUMÉRICA
# =============================================================================

_NUMERO = re.compile(r"(\d+[.,]?\d*)")


def extraer_numero(x):

    if pd.isna(x):
        return np.nan

    m = _NUMERO.search(str(x))

    if m is None:
        return np.nan

    return float(m.group(1).replace(",", "."))


def extraer_numeros(df):

    df = df.copy()

    for col in ["precio","peso","grueso","n_paginas"]:
        if col in df.columns:
            df[col] = df[col].apply(extraer_numero)

    # dimensiones (ej: 240 x 170 mm)
    medidas = df["dimensiones"].astype(str).str.extract(r"(\d+[.,]?\d*)\D+(\d+[.,]?\d*)")

    df["alto_mm"] = medidas[0].str.replace(",", ".", regex=False).astype(float)
    df["ancho_mm"] = medidas[1].str.replace(",", ".", regex=False).astype(float)

    return df


# =============================================================================
# FEATURES
# =============================================================================

def crear_columnas(df):

    df = df.copy()

    # Flags
    df["es_ilustrado"] = (
        df[ILUSTRACIONES]
        .apply(lambda col: col.str.len())
        .sum(axis=1)
        > 0
    )

    escolar = (
        df[ESCOLARES]
        .apply(lambda col: col.str.len())
        .sum(axis=1)
        > 0
    )

    comentada = (
        df["otros_contribuidores"]
        .str.len()
        > 0
    )

    adaptada = (
        df["adaptacion"]
        .str.len()
        > 0
    )

    ibd = (
        df["ibd"]
        .str.len()
        > 0
    )


    # Tipo edición
    df["tipo_edicion"] = 'Edición normal'

    df.loc[escolar, "tipo_edicion"] = "escolar"
    df.loc[comentada & ~escolar,"tipo_edicion"] = "comentada"
    df.loc[adaptada,"tipo_edicion"] = "adaptada"
    df.loc[ibd,"tipo_edicion"] = "ibd"

    # Presentación
    titulo = df["titulo"].fillna("").str.lower()

    df["presentacion"] = "tomo único"
    df.loc[titulo.str.contains(r"\bestuche\b",regex=True,na=False),"presentacion"] = "estuche"
    df.loc[titulo.str.contains(r"obras?\s+completas?",regex=True,na=False),"presentacion"] = "obras completas"
    df.loc[titulo.str.contains(r"\bpack\b",regex=True,na=False),"presentacion"] = "pack"
    df.loc[titulo.str.contains(r"\bvol?\s\b",regex=True,na=False),"presentacion"] = "colección"
    df.loc[titulo.str.contains(r"\btomo?\s\b",regex=True,na=False),"presentacion"] = "colección"

    # URL imagen
    ean = df["ean"].astype(str)
    df["img"] = (
        "https://static.cegal.es/imagenes/marcadas/"
        + ean.str[:8]
        + "/"
        + ean
        + ".gif"
    )

    return df


def definir_aparato_critico(df, cols_ap):
    mask = df[cols_ap].notna().any(axis=1)

    df["aparato_critico"] = mask

    df["tipo_aparato_critico"] = df[cols_ap].apply(lambda fila: [col for col in cols_ap if pd.notna(fila[col])],axis=1)

    df.loc[~mask, "tipo_aparato_critico"] = np.nan

    return df

# =============================================================================
# RELLENO DE VALORES
# =============================================================================

def moda(x):
    """
    Devuelve la moda de una serie.
    """

    x = x.dropna()

    if len(x) == 0:
        return np.nan

    return x.mode().iloc[0]


def rellenar_columnas(df):

    df = df.copy()

    # Medidas por editorial + colección
    for col in ["alto_mm", "ancho_mm", "precio", "n_paginas"]:
        mediana_col = df.groupby(["editorial", "coleccion"])[col].transform("median")
        mediana_enc = df.groupby(["editorial", "encuadernacion"])[col].transform("median")
        
        df[col] = df[col].fillna(mediana_col).fillna(mediana_enc)


    # Grosor
    df["grueso"] = df["grueso"].fillna(df["n_paginas"] * 0.04)

    # Peso
    peso_estimado = df['peso'].fillna(df["alto_mm"] * df["ancho_mm"] * df["n_paginas"] * 0.08 + 120)

    df["peso"] = df["peso"].fillna(peso_estimado)

    # Idioma original
    df["autor_principal"] = (
        df["autoria"]
        .apply(
            lambda x:
                x[0]
                if len(x)
                else np.nan
        )
    )

    idioma = (
        df.groupby("autor_principal")[
            "idioma_original"
        ]
        .transform(moda)
    )

    df["idioma_original"] = df["idioma_original"].fillna(idioma)

    df.drop(columns="autor_principal", inplace=True)

    return df


# =============================================================================
# LIMPIEZA FINAL
# =============================================================================

def limpiar_columnas(df):

    borrar = (
        TRADUCTOR_EDITOR
        + OTROS_CONTRIBUIDORES
        + ILUSTRACIONES
    )

    borrar = [c for c in borrar if c in df.columns]

    return df.drop(columns=borrar)


# =============================================================================
# PIPELINE
# =============================================================================

def limpiar_df_completa(data, dict_editoriales=None, dict_encuadernacion=None):

    df = data.copy()

    # Normalización
    df = normalizar_columnas_lista(df)

    # Limpieza
    df = limpieza_basica(df, dict_editoriales, dict_encuadernacion)

    # Merge colaboradores
    df = merge_columnas(df, "traductor_y_editor", TRADUCTOR_EDITOR)
    df = merge_columnas(df,"otros_contribuidores", OTROS_CONTRIBUIDORES)
    df = merge_columnas(df, "subcategorias", CATEGORIAS)

    # Feature engineering
    df = contador_nombres(df)
    df = extraer_numeros(df)
    df = crear_columnas(df)
    df = definir_aparato_critico(df, OTROS_CONTRIBUIDORES)

    # Relleno
    df = rellenar_columnas(df)

    # Limpieza final
    df = limpiar_columnas(df)

    df = df[[c for c in COLUMNAS_FINALES if c in df.columns]]

    return df

data_clean = limpiar_df_completa(data_raw)

### 3. Merge de los datos del SPI

In [ ]:
# ======================================================================================
# DATOS DE SPI
# ======================================================================================

import json
import pandas as pd 
import numpy as np 
from pathlib import Path

def merge_spi(ruta_spi="data/bronze/spi", ruta_dict = 'data/json/spi_a_ed.json'):
    with open(ruta_dict, "r", encoding="utf-8") as f:
        spi_a_ed = json.load(f)

    rutas = sorted(Path(ruta_spi).glob("*.csv"))
    dfs = []
    for ruta in rutas:
        df = pd.read_csv(ruta)

        if "Editorial" not in df.columns:
            raise ValueError(f"{ruta.name} no contiene la columna 'Editorial'.")

        nombre = ruta.stem.lower().replace("clasificacion_", "")

        df = df.rename(columns={
            c: f"{c}_{nombre}"
            for c in df.columns
            if c != "Editorial"
        })
        df["Editorial"] = df["Editorial"].map(spi_a_ed)
        df = (
            df
            .groupby("Editorial", as_index=False)
            .first()
        )
        df = df.set_index("Editorial")
        dfs.append(df)
    
    df = pd.concat(dfs, axis=1, join="outer").reset_index()

    df_selection = df[df['Editorial'].isin(spi_a_ed.values())].copy()

    return df_selection

def prestigio_editorial(df):
    df_prestigio = pd.DataFrame({})
    df_prestigio["Editorial"] = df["Editorial"]

    for i in range(1, len(df.columns) - 2, 2):
        col_pos = df.columns[i]
        col_icee = df.columns[i + 1]

        nombre_col = f"prestigio_{str(col_pos).replace('Posición', '').strip()}"

        mask = df[[col_pos, col_icee]].notna().all(axis=1)

        icee = df[col_icee]
        icee_norm = (icee - icee.min()) / (icee.max() - icee.min())

        n = df[col_pos].max()
        percentil = 1 - (df[col_pos] - 1) / (n - 1) if n > 1 else pd.Series(1.0, index=df.index)

        df_prestigio[nombre_col] = 0.0
        df_prestigio.loc[mask, nombre_col] = 0.1 + 0.9*(0.8 * icee_norm[mask] + 0.2 * percentil[mask])

    return df_prestigio

datos_spi = merge_spi()
datos_spi_prestigio = prestigio_editorial(datos_spi)

Fin de la capa **silver**